# Experiment 05 — CMB Radon Scar Scan

Validates **RBLE Eq. (6)** spherical Radon scar on $S^2$:

$$S_{\mathrm{RBLE}}(\hat{n}) = \mathcal{R}_{S^2}[\Delta T](\hat{n}, \eta)$$

Synthetic HEALPix map with injected anisotropic scar $\to$ `radon_transform_s2` $\to$ `compute_rble_signature` $\to$ Mollweide sky plot.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "polomni").is_dir():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import matplotlib.pyplot as plt

try:
    import networkx as nx
except ImportError:
    nx = None

%matplotlib inline
plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 11})
print(f"polomni root: {ROOT}")


## Generate synthetic isotropic CMB map


In [ ]:
from polomni.observatory.ingest.healpix_loader import synthetic_cmb_map
from polomni.observatory.scoring.rble_signature import (
    compute_rble_signature,
    inject_synthetic_scar,
)
from polomni.core.radon.transform_s2 import radon_transform_s2

NSIDE = 32
SEED = 42
cmb = synthetic_cmb_map(NSIDE, seed=SEED)
assert cmb.ndim == 1
assert cmb.size == 12 * NSIDE * NSIDE
print(f"CMB map: nside={NSIDE}, npix={cmb.size}, RMS={np.std(cmb):.4f}")


## Inject known Radon scar along axis $\hat{n} = (0,0,1)$


In [ ]:
true_axis = np.array([0.0, 0.0, 1.0])
AMPLITUDE = 8.0
scarred = inject_synthetic_scar(cmb, true_axis, amplitude=AMPLITUDE)

delta = scarred - cmb
assert np.max(np.abs(delta)) > 0.5
print(f"Scar amplitude max |delta T| = {np.max(np.abs(delta)):.4f}")


## Spherical Radon transform along scar axis


In [ ]:
eta_vals = np.linspace(0, 2 * np.pi, 24, endpoint=False)
radon_vals = [radon_transform_s2(scarred, true_axis, float(eta), nside=NSIDE) for eta in eta_vals]
radon_vals = np.asarray(radon_vals)
assert radon_vals.size == eta_vals.size
print(f"Radon S2 values: min={radon_vals.min():.4f}, max={radon_vals.max():.4f}")


## RBLE signature scoring


In [ ]:
report_iso = compute_rble_signature(cmb, n_hat=true_axis)
report_scar = compute_rble_signature(scarred, n_hat=true_axis)

assert report_scar.rble_score >= report_iso.rble_score
print("Isotropic score:", report_iso.rble_score)
print("Scarred score:", report_scar.rble_score)
print("Falsification flags:", report_scar.falsification_flags)


## Preferred-axis search (auto scan)


In [ ]:
auto_report = compute_rble_signature(scarred, n_hat=None, scan_angles=18)
pref = np.asarray(auto_report.preferred_axis)
assert pref.shape == (3,)
assert np.isclose(np.linalg.norm(pref), 1.0, atol=1e-6)
print("Auto preferred axis:", np.round(pref, 3))
print("Auto RBLE score:", auto_report.rble_score)


## Mollweide sky plot (healpy or fallback)


In [ ]:
try:
    import healpy as hp

    hp.mollview(scarred, title="Scarred CMB (Mollweide)", unit="muK", sub=(1, 1, 1))
    hp.graticule()
    plt.show()
except ImportError:
    # Fallback equirectangular projection
    nside = NSIDE
    npix = cmb.size
    theta = np.linspace(0, np.pi, 2 * nside)
    phi = np.linspace(0, 2 * np.pi, 4 * nside)
    grid = scarred[: 2 * nside * 4 * nside].reshape(2 * nside, 4 * nside)
    fig, ax = plt.subplots(subplot_kw={"projection": "mollweide"})
    lon = phi - np.pi
    lat = np.pi / 2 - theta
    LON, LAT = np.meshgrid(lon, lat)
    ax.pcolormesh(LON, LAT, grid, cmap="RdBu_r", shading="auto")
    ax.set_title("Scarred CMB (fallback Mollweide)")
    plt.tight_layout()
    plt.show()


## Scar residual map


In [ ]:
try:
    import healpy as hp

    hp.mollview(delta, title="Injected scar residual delta T", cmap="hot", sub=(1, 1, 1))
    plt.show()
except ImportError:
    fig, ax = plt.subplots()
    ax.hist(delta, bins=50, color="#c0392b", alpha=0.8)
    ax.set_title("Scar residual histogram (healpy unavailable)")
    plt.tight_layout()
    plt.show()


## Radon S2 vs eta


In [ ]:
fig, ax = plt.subplots()
ax.plot(eta_vals, radon_vals, "o-", color="#d35400")
ax.set_xlabel("eta (radians along great circle)")
ax.set_ylabel("R_S2[map](n_hat, eta)")
ax.set_title("Spherical Radon transform of scarred map")
plt.tight_layout()
plt.show()


## Assert scar detection exceeds isotropic baseline


In [ ]:
assert report_scar.rble_score > report_iso.rble_score
assert report_scar.falsification_flags.get("radon_anisotropic", False)
print("Scar detection exceeds isotropic null.")


## Score comparison bar chart


In [ ]:
fig, ax = plt.subplots()
ax.bar(["isotropic", "scarred (fixed n)", "scarred (auto)"],
       [report_iso.rble_score, report_scar.rble_score, auto_report.rble_score],
       color=["#95a5a6", "#27ae60", "#2980b9"])
ax.axhline(1.2, color="red", ls="--", label="anisotropy threshold")
ax.set_ylabel("S_RBLE score")
ax.set_title("RBLE signature detection strength")
ax.legend()
plt.tight_layout()
plt.show()


## Conclusions

1. Synthetic scar injection produces a localized anisotropic feature on the HEALPix sphere.
2. `radon_transform_s2` varies along the great-circle parameter $\eta$, confirming directional sensitivity.
3. `compute_rble_signature` scores the scarred map higher than the isotropic null — recovery of injected structure.
4. Mollweide visualization localizes the scar along the injected axis $\hat{z}$, supporting observatory use of RBLE Eq. (6).
